# 03_map_to_zones
Maps taxi, FHV, Citi Bike and traffic data to taxi zones. Saves `trips_zoned` and the traffic tables.

In [ ]:
%pip install shapely pyproj pyshp

In [ ]:
SCHEMA = "workspace.default"
YEAR = 2025
CITIBIKE_TABLE = f"{SCHEMA}.citibike_clean"
TRAFFIC_TABLE = f"{SCHEMA}.traffic_counts_clean"
YELLOW_TABLE = f"{SCHEMA}.yellow_clean"  # Dev's tables
FHV_TABLE = f"{SCHEMA}.fhv_clean"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip"
LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
BIKE_SNAP_FT = 300  # pier stations
TRAFFIC_SNAP_FT = 1000  # bridges

In [ ]:
# taxi zones
import io, tempfile, zipfile
import numpy as np
import pandas as pd
import requests, shapefile, shapely
from pyproj import Transformer
from shapely.geometry import shape
from pyspark.sql import functions as F

tmp = tempfile.mkdtemp()
zipfile.ZipFile(io.BytesIO(requests.get(ZONES_URL, timeout=120).content)).extractall(tmp)
sf = shapefile.Reader(f"{tmp}/taxi_zones/taxi_zones.shp")
fields = [f[0] for f in sf.fields[1:]]
zone_ids = np.array([int(dict(zip(fields, r))["LocationID"]) for r in sf.records()])
zone_geoms = shapely.make_valid(np.array([shape(s.__geo_interface__) for s in sf.shapes()]))
tree = shapely.STRtree(zone_geoms)
to_ft = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True)

def zone_of(x, y, snap_ft=0):
    pts = shapely.points(x, y)
    pt, z = tree.query(pts, predicate="intersects")
    out = np.full(len(pts), -1)
    out[pt] = zone_ids[z]
    miss = np.flatnonzero(out < 0)
    if snap_ft and len(miss):  # snap nearby
        i, z = tree.query_nearest(pts[miss], max_distance=snap_ft)
        out[miss[i]] = zone_ids[z]
    return out

print(len(zone_ids), "polygons,", len(set(zone_ids)), "zones")

In [ ]:
# sanity check
check = pd.DataFrame({"place": ["Times Sq", "Central Park", "JFK", "Hudson River"],
                      "lat": [40.7580, 40.7812, 40.6413, 40.7700],
                      "lng": [-73.9855, -73.9665, -73.7781, -74.0080]})
check["zone"] = zone_of(*to_ft.transform(check.lng.values, check.lat.values))
print(check)

In [ ]:
# zone lookup
lookup = spark.createDataFrame(pd.read_csv(LOOKUP_URL, keep_default_na=False)).select(
    F.col("LocationID").cast("int").alias("zone"),
    F.col("Borough").alias("borough"),
    F.col("Zone").alias("zone_name"),
    "service_zone")
lookup.write.mode("overwrite").saveAsTable(f"{SCHEMA}.taxi_zone_lookup")

In [ ]:
# citi bike
cb = (spark.table(CITIBIKE_TABLE)
      .withColumn("s_lat", F.round("start_lat", 5)).withColumn("s_lng", F.round("start_lng", 5))
      .withColumn("e_lat", F.round("end_lat", 5)).withColumn("e_lng", F.round("end_lng", 5)))
pts = (cb.select(F.col("s_lat").alias("lat"), F.col("s_lng").alias("lng"))
         .union(cb.select("e_lat", "e_lng")).distinct().toPandas())
pts["zone"] = zone_of(*to_ft.transform(pts.lng.values, pts.lat.values), snap_ft=BIKE_SNAP_FT)
print(len(pts), "distinct points,", (pts.zone < 0).sum(), "outside every zone")
pts = spark.createDataFrame(pts[pts.zone > 0]).withColumn("zone", F.col("zone").cast("int"))
cbz = (cb.join(F.broadcast(pts.toDF("s_lat", "s_lng", "start_zone")), ["s_lat", "s_lng"])
         .join(F.broadcast(pts.toDF("e_lat", "e_lng", "end_zone")), ["e_lat", "e_lng"])
         .drop("s_lat", "s_lng", "e_lat", "e_lng"))
n0 = cb.count()
cbz.write.mode("overwrite").saveAsTable(f"{SCHEMA}.citibike_zoned")
n1 = spark.table(f"{SCHEMA}.citibike_zoned").count()
print(n0, n1)

In [ ]:
# traffic
tr = spark.table(TRAFFIC_TABLE)
loc = tr.select("x_ft", "y_ft").distinct().toPandas()
loc["zone"] = zone_of(loc.x_ft.values, loc.y_ft.values, snap_ft=TRAFFIC_SNAP_FT)
print(len(loc), "counter locations,", (loc.zone < 0).sum(), "outside every zone")
loc = spark.createDataFrame(loc[loc.zone > 0]).withColumn("zone", F.col("zone").cast("int"))
trz = tr.join(F.broadcast(loc), ["x_ft", "y_ft"])
trz.write.mode("overwrite").saveAsTable(f"{SCHEMA}.traffic_zoned")

In [ ]:
# hourly volume
seg_hour = (spark.table(f"{SCHEMA}.traffic_zoned")
            .groupBy("zone", "segment_id", "direction", "date", "hour", "dow")
            .agg(F.sum("vol").alias("vol"), F.count("*").alias("n_15min"))
            .filter("n_15min = 4")
            .withColumn("is_weekday", F.col("dow").between(2, 6))
            .join(spark.table(f"{SCHEMA}.taxi_zone_lookup").select("zone", "borough"), "zone"))

def hourly(keys):
    return (seg_hour.groupBy(*keys, "is_weekday", "hour")
            .agg(F.avg("vol").alias("avg_vol_per_hour"),
                 F.countDistinct("segment_id", "direction").alias("n_counters"),
                 F.count("*").alias("n_obs")))

hourly(["zone", "borough"]).write.mode("overwrite").saveAsTable(f"{SCHEMA}.traffic_zone_hourly")
hourly(["borough"]).write.mode("overwrite").saveAsTable(f"{SCHEMA}.traffic_borough_hourly")
spark.table(f"{SCHEMA}.traffic_borough_hourly").filter("is_weekday AND hour BETWEEN 16 AND 18").orderBy("borough", "hour").show()

In [ ]:
# all trips
def trips(mode, df, pu, do, start, end):
    return df.select(F.lit(mode).alias("mode"),
                     F.col(pu).cast("int").alias("pu_zone"),
                     F.col(do).cast("int").alias("do_zone"),
                     F.col(start).cast("timestamp").alias("pickup_ts"),
                     F.col(end).cast("timestamp").alias("dropoff_ts"))

parts = [trips("bike", spark.table(f"{SCHEMA}.citibike_zoned"), "start_zone", "end_zone", "started_at", "ended_at")]
for mode, table, start, end in [("taxi", YELLOW_TABLE, "pickup_datetime", "dropoff_datetime"),
                                ("fhv", FHV_TABLE, "pickup_datetime", "dropoff_datetime")]:
    if spark.catalog.tableExists(table):
        parts.append(trips(mode, spark.table(table), "PULocationID", "DOLocationID", start, end))
    else:
        print(f"skipping {mode}: {table} not found")

all_trips = parts[0]
for p in parts[1:]:
    all_trips = all_trips.unionByName(p)
all_trips = (all_trips
    .withColumn("minutes", (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60)
    .filter(F.col("pu_zone").between(1, 263) & F.col("do_zone").between(1, 263)
            & F.col("minutes").between(1, 180) & (F.year("pickup_ts") == YEAR))
    .withColumn("hour", F.hour("pickup_ts"))
    .withColumn("dow", F.dayofweek("pickup_ts"))
    .withColumn("month", F.month("pickup_ts")))
all_trips.write.mode("overwrite").partitionBy("mode").saveAsTable(f"{SCHEMA}.trips_zoned")
spark.table(f"{SCHEMA}.trips_zoned").groupBy("mode").count().show()